In [1]:
# Cell 1: Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import requests
from fredapi import Fred
import sys
import os

In [2]:
# Cell 2: Load API key from config.py and connect to FRED
sys.path.append('..')  
from config import FRED_API_KEY

# Connect to FRED using API key
fred = Fred(api_key=FRED_API_KEY)

print("Connected to FRED successfully :)")

Connected to FRED successfully :)


In [3]:
# Cell 3: Download economic data from FRED
# Brent Crude Oil price (USD per barrel) - a key indicator of energy inflation
oil = fred.get_series('DCOILBRENTEU', observation_start='2000-01-01')

# GBP/USD exchange rate - affects import prices, which feed into inflation
gbp_usd = fred.get_series('DEXUSUK', observation_start='2000-01-01')

# US CPI - useful leading indicator since US CPI data releases before UK
us_cpi = fred.get_series('CPIAUCSL', observation_start='2000-01-01')

# Check number of observations and latest figures
print(f"Oil price observations: {len(oil)}")
print(f"GBP/USD observations: {len(gbp_usd)}")
print(f"US CPI observations: {len(us_cpi)}")

print("\nLatest oil price:", round(oil.iloc[-1], 2), "USD/barrel")
print("Latest GBP/USD rate:", round(gbp_usd.iloc[-1], 4))

Oil price observations: 6988
GBP/USD observations: 6990
US CPI observations: 321

Latest oil price: 126.55 USD/barrel
Latest GBP/USD rate: 1.333


In [4]:
# Cell 4: Preview data for sanity check
print("=== OIL PRICE (last 5 observations) ===")
print(oil.tail())

print("\n=== GBP/USD RATE (last 5 observations) ===")
print(gbp_usd.tail())

=== OIL PRICE (last 5 observations) ===
2026-10-08    130.18
2026-10-09    130.03
2026-10-12    129.95
2026-10-13    125.78
2026-10-14    126.55
dtype: float64

=== GBP/USD RATE (last 5 observations) ===
2026-10-12       NaN
2026-10-13    1.3255
2026-10-14    1.3155
2026-10-15    1.3257
2026-10-16    1.3330
dtype: float64


In [5]:
# Cell 5: Save raw data to data/raw/ folder
oil.to_csv('../data/raw/brent_crude.csv', header=['oil_price'])
gbp_usd.to_csv('../data/raw/gbp_usd.csv', header=['gbp_usd'])
us_cpi.to_csv('../data/raw/us_cpi.csv', header=['us_cpi'])

print("\nAll raw data saved to data/raw/ folder :)")


All raw data saved to data/raw/ folder :)


In [6]:
# Check what sheets are inside the consumerpriceinflationdetailedreferencetables Excel file
import openpyxl

wb = openpyxl.load_workbook('../data/raw/consumerpriceinflationdetailedreferencetables.xlsx', read_only=True)
print("Sheets in ONS Download:")
for name in wb.sheetnames:
    print(" -", name)

Sheets in ONS Download:
 - data


C:\Users\westo\anaconda3\envs\inflation_model\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


In [7]:
# Load new ONS file (restructured as single 'data' sheet from August 2026 release)
df_ons = pd.read_excel('../data/raw/consumerpriceinflationdetailedreferencetables.xlsx', 
                       sheet_name='data', header=None)

# Find D7G7 column (CPI annual % change)
cdid_row = df_ons.iloc[1]
d7g7_col = None
for col_idx, val in enumerate(cdid_row):
    if str(val).strip() == 'D7G7':
        d7g7_col = col_idx

print(f"D7G7 found in column {d7g7_col}")

# Extract CPI series
cpi_raw = df_ons.iloc[6:, [0, d7g7_col]].copy()
cpi_raw.columns = ['date', 'cpi']

# Keep only monthly rows
monthly_mask = cpi_raw['date'].astype(str).str.match(
    r'^\d{4}\s(JAN|FEB|MAR|APR|MAY|JUN|JUL|AUG|SEP|OCT|NOV|DEC)$', na=False
)
cpi_raw = cpi_raw[monthly_mask]

# Convert dates and values
cpi_raw['date'] = pd.to_datetime(cpi_raw['date'], format='%Y %b')
cpi_raw['cpi'] = pd.to_numeric(cpi_raw['cpi'], errors='coerce')
uk_cpi = cpi_raw.set_index('date').sort_index().dropna()

# Save
uk_cpi.to_csv('../data/raw/uk_cpi.csv')

print(f"CPI observations: {len(uk_cpi)}")
print(f"Date range: {uk_cpi.index[0].date()} to {uk_cpi.index[-1].date()}")
print("\nLatest 5 readings:")
print(uk_cpi.tail())

D7G7 found in column 879
CPI observations: 452
Date range: 1989-01-01 to 2026-08-01

Latest 5 readings:
            cpi
date           
2026-04-01  2.8
2026-05-01  2.8
2026-06-01  2.6
2026-07-01  2.9
2026-08-01  3.1


In [8]:
# Save clean CPI data
uk_cpi.to_csv('../data/raw/uk_cpi.csv')
print("UK CPI saved :)")

UK CPI saved :)


In [9]:
# UK average earnings index from FRED
uk_wages_fred = fred.get_series('LCEAMN01GBM661S', observation_start='2000-01-01')

# Convert index to annual % change (year on year)
uk_wages = uk_wages_fred.pct_change(12) * 100
uk_wages = uk_wages.dropna()

# Save to raw folder
uk_wages.to_csv('../data/raw/uk_wages.csv', header=['wages'])

# Check wages observations and preview latest figures
print(f"Wages observations: {len(uk_wages)}")
print(f"Date range: {uk_wages.index[0].date()} to {uk_wages.index[-1].date()}")
print("\nLatest 5 readings:")
print(uk_wages.tail())

Wages observations: 307
Date range: 2001-01-01 to 2026-07-01

Latest 5 readings:
2026-03-01    3.535417
2026-04-01    4.861151
2026-05-01    2.519524
2026-06-01    2.542015
2026-07-01    2.411324
dtype: float64


In [10]:
# UK unemployment rate from FRED
uk_unemp_fred = fred.get_series('LRHUTTTTGBM156S', observation_start='2000-01-01')

# Save to raw folder
uk_unemp_fred.to_csv('../data/raw/uk_unemployment.csv', header=['unemployment'])

# Check unemployment observations and preview latest figures
print(f"Unemployment observations: {len(uk_unemp_fred)}")
print(f"Date range: {uk_unemp_fred.index[0].date()} to {uk_unemp_fred.index[-1].date()}")
print("\nLatest 5 readings:")
print(uk_unemp_fred.tail())

print("Wages and unemployment saved!")

Unemployment observations: 318
Date range: 2000-01-01 to 2026-06-01

Latest 5 readings:
2026-02-01    5.0
2026-03-01    4.9
2026-04-01    4.9
2026-05-01    4.9
2026-06-01    4.9
dtype: float64
Wages and unemployment saved!


In [11]:
# Check data collection
print("\n=== DATA COLLECTION COMPLETE ===")
print(f"UK CPI:          {len(uk_cpi)} monthly observations, up to {uk_cpi.index[-1].date()}")
print(f"UK Wages:        {len(uk_wages)} monthly observations, up to {uk_wages.index[-1].date()}")
print(f"UK Unemployment: {len(uk_unemp_fred)} monthly observations, up to {uk_unemp_fred.index[-1].date()}")
print(f"Brent Crude Oil: {len(oil)} daily observations, up to {oil.index[-1].date()}")
print(f"GBP/USD:         {len(gbp_usd)} daily observations, up to {gbp_usd.index[-1].date()}")
print(f"US CPI:          {len(us_cpi)} monthly observations, up to {us_cpi.index[-1].date()}")


=== DATA COLLECTION COMPLETE ===
UK CPI:          452 monthly observations, up to 2026-08-01
UK Wages:        307 monthly observations, up to 2026-07-01
UK Unemployment: 318 monthly observations, up to 2026-06-01
Brent Crude Oil: 6988 daily observations, up to 2026-10-14
GBP/USD:         6990 daily observations, up to 2026-10-16
US CPI:          321 monthly observations, up to 2026-09-01
